In [1]:
!pip install -q transformers pandas numpy torch

from transformers import AutoModel, CLIPTextModel
import json
import pandas as pd
import transformers
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.4 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [2]:
# import json
# import pandas as pd
# import transformers
# from transformers import AutoTokenizer
# from tqdm.notebook import tqdm  # Changed to the notebook-optimized version

# # 1. Mute the verbose transformers warnings so they don't break our UI
# transformers.logging.set_verbosity_error()

# print("\n--- Starting Lightning-Fast arXiv Processing ---")
# print("Loading tokenizer (CLIP - for accurate 77-token measurement)...")
# tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-large-patch14")

# # The precise Kaggle mount path for the flat file
# file_path = "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"

# ready_datapoints = []
# target_samples = 2000000 

# print(f"Extracting data points until target of {target_samples} is hit...")
# pbar = tqdm(total=target_samples, desc="Extracting Sequences", unit="seq")

# # Read line-by-line to protect RAM and bypass filesystem lag
# with open(file_path, 'r', encoding='utf-8') as f:
#     for line in f:
#         try:
#             data = json.loads(line)
            
#             # Extract and clean up the paper title
#             title = data.get('title', '').strip()
#             title = " ".join(title.split())
            
#             # Extract abstract and split it up into individual sentences
#             abstract = data.get('abstract', '').strip()
#             abstract = " ".join(abstract.split())
#             sentences = abstract.split('. ')
            
#             # Combine both titles and sentences as unique text candidates
#             candidates = [title] + sentences
            
#             for text in candidates:
#                 # 1st Filter: Fast character check to eliminate outliers quickly
#                 if 30 < len(text) < 400:
                    
#                     # 2nd Filter: Strict CLIP token calculation
#                     token_count = len(tokenizer.tokenize(text))
                    
#                     # Perfect bounds check for LanguageBind alignment
#                     if 10 < token_count <= 77:
#                         ready_datapoints.append(text)
#                         pbar.update(1)
                        
#                 if len(ready_datapoints) >= target_samples:
#                     break
                    
#         except Exception:
#             continue  # Gracefully ignore any malformed metadata lines
            
#         if len(ready_datapoints) >= target_samples:
#             break

# pbar.close()

# print("\n--- Extraction Complete ---")
# print(f"Successfully generated {len(ready_datapoints)} short academic training sequences.")

# # ==========================================
# # Save Output
# # ==========================================
# output_path = "/kaggle/working/ready_training_data.parquet"
# final_df = pd.DataFrame({'text': ready_datapoints})
# final_df.to_parquet(output_path)

# print(f"Data saved successfully to: {output_path}")
# print("Process Finished! Your structured text tensors are isolated and ready.")

In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ==========================================
# 1. Custom Text Dataset Class
# ==========================================
class TextCorpusDataset(Dataset):
    def __init__(self, parquet_path):
        print(f"Loading {parquet_path} into memory...")
        # Read only the text column to keep memory usage light
        df = pd.read_parquet(parquet_path, columns=['text'])
        self.texts = df['text'].tolist()
        print(f"Dataset initialized with {len(self.texts):,} samples.")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Return the raw string. We defer tokenization to the collate function.
        return self.texts[idx]


# ==========================================
# 2. Optimized Parallel Tokenizer Class
# ==========================================
class DualModelCollate:
    def __init__(self):
        # Initialize both tokenizers
        print("Initializing Dual Tokenizers...")
        self.teacher_tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-large-patch14")
        self.student_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5",trust_remote_code=True)

    def __call__(self, batch_strings):
        """
        Takes a list of raw strings from the DataLoader and tokenizes them 
        simultaneously using highly optimized Hugging Face C++ backends.
        """
        # Teacher Tokenization (CLIP / LanguageBind format)
        teacher_inputs = self.teacher_tokenizer(
            batch_strings,
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )
        
        # Student Tokenization (Nomic format)
        student_inputs = self.student_tokenizer(
            batch_strings,
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )
        
        # Pass the raw strings along just in case you need them for logging/debugging
        return {
            "raw_text": batch_strings,
            "teacher": teacher_inputs,  # Dictionary containing input_ids, attention_mask
            "student": student_inputs   # Dictionary containing input_ids, attention_mask
        }

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

# ==========================================
# 1. The Projection MLP (Student Head)
# ==========================================
class ProjectionMLP(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=2048, output_dim=768):
        super().__init__()
        # A standard bottleneck architecture for embedding alignment
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.mlp(x)

# ==========================================
# 2. Custom Cosine Similarity Loss
# ==========================================
class CosineLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, student_preds, teacher_targets):
        # Calculate cosine similarity (-1 to 1) along the feature dimension (dim=1)
        cos_sim = F.cosine_similarity(student_preds, teacher_targets, dim=1)
        
        # L = 1 - cosine_sim
        # If perfectly aligned (1), loss becomes 0. If opposite (-1), loss is 2.
        loss = 1.0 - cos_sim
        
        # Return the mean loss across the batch
        return loss.mean()

class ProjectionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [5]:
import torch

def save_mlp_model(model, filepath="/kaggle/working/nomic_to_languagebind_mlp.pth"):
    """
    Saves the trained MLP weights to the specified path.
    Locks the model in eval mode before saving to ensure dropout states are stable.
    """
    model.eval() 
    torch.save(model.state_dict(), filepath)
    print(f"[SAVE] Model weights successfully saved to: {filepath}")


def load_mlp_model(filepath="/kaggle/working/nomic_to_languagebind_mlp.pth", input_dim=768, hidden_dim=2048, output_dim=768):
    """
    Instantiates a fresh ProjectionMLP and loads the saved weights onto the appropriate device.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Assumes ProjectionMLP class is already declared in your notebook
    model = ProjectionModel(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim).to(device)
    
    # Load weights
    model.load_state_dict(torch.load(filepath, map_location=device))
    
    print(f"[LOAD] Model successfully loaded from: {filepath}")
    return model

In [6]:
# PARQUET_PATH = "/kaggle/working/ready_training_data.parquet"
# BATCH_SIZE = 128
# EPOCHS = 5

# print("\n--- Initializing PyTorch Data Pipeline ---")
# dataset = TextCorpusDataset(PARQUET_PATH)

# # 2. Initialize Collate Function (The Parallel Tokenizers)
# dual_collate_fn = DualModelCollate()

# # 3. Create DataLoader
# dataloader = DataLoader(
#     dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=2,
#     collate_fn=dual_collate_fn,
#     pin_memory=True
# )

# print("\n--- Starting Training Process ---")

# nomic_base = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
# languagebind_base = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14")
# # --- Setup Device ---
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Training on device: {device}")

# # Move base models to device
# nomic_base.to(device)
# languagebind_base.to(device)

# # Freezing base models completely
# nomic_base.eval()
# languagebind_base.eval()
# for param in nomic_base.parameters():
#     param.requires_grad = False
# for param in languagebind_base.parameters():
#     param.requires_grad = False

# # --- Initialize Your Custom Architecture & Loss ---
# model = ProjectionModel(input_dim=768, hidden_dim=2048, output_dim=768).to(device)
# criterion = CosineLoss()
# optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# model.train()

# # --- Execution Loop ---
# print("Beginning alignment iteration batches...")
# for epoch in range(EPOCHS):
#     total_loss = 0.0
    
#     # Wrap dataloader with tqdm for a clean notebook progress bar
#     progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
#     for batch_idx, batch in progress_bar:
#         # Move tokenized inputs to the active compute hardware
#         student_inputs = {k: v.to(device) for k, v in batch['student'].items()}
#         teacher_inputs = {k: v.to(device) for k, v in batch['teacher'].items()}
        
#         # 1. Forward pass through frozen base models to collect source tensors
#         with torch.no_grad():
#             # Extract dense embeddings from CLS tokens
#             nomic_vectors = nomic_base(**student_inputs).last_hidden_state[:, 0, :]
#             teacher_vectors = languagebind_base(**teacher_inputs).last_hidden_state[:, 0, :]
            
#             # Normalize target teacher embeddings to the spherical unit surface
#             teacher_vectors = F.normalize(teacher_vectors, p=2, dim=1)
            
#         # 2. Map student embeddings through the projection head
#         student_preds = model(nomic_vectors)
#         student_preds = F.normalize(student_preds, p=2, dim=1)
        
#         # 3. Compute loss using your 1 - cosine_sim implementation
#         loss = criterion(student_preds, teacher_vectors)
        
#         # 4. Weight update optimization step
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
        
#         current_loss = loss.item()
#         total_loss += current_loss
        
#         # Dynamically update the tqdm postfix instead of using print statements to avoid breaking the bar layout
#         if batch_idx % 10 == 0:
#             progress_bar.set_postfix({"batch_loss": f"{current_loss:.4f}"})
            
#     avg_loss = total_loss / len(dataloader)
#     print(f"=== Epoch {epoch+1} Summary | Global Average Loss: {avg_loss:.4f} ===")

# print("\n--- Training Process Complete ---")

# save_mlp_model(model,"/kaggle/working/nomic_to_languagebind_mlp.pth")
# print("\n--- Saved model to /kaggle/working/nomic_to_languagebind_mlp.pth ---")

In [7]:
import torch
import numpy as np
from transformers import AutoModel, CLIPTextModel
from tqdm.notebook import tqdm

PARQUET_PATH = "/kaggle/working/ready_training_data.parquet"
BATCH_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Extracting features on: {device}")

# Load models for one-time extraction
nomic_base = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True).to(device)
languagebind_base = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device)

nomic_base.eval()
languagebind_base.eval()

print("\n--- Initializing PyTorch Data Pipeline ---")
dataset = TextCorpusDataset(PARQUET_PATH)

# 2. Initialize Collate Function (The Parallel Tokenizers)
dual_collate_fn = DualModelCollate()

# 3. Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=dual_collate_fn,
    pin_memory=True
)

nomic_list = []
teacher_list = []

print("Extracting frozen embeddings to disk (One-time pass)...")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Extracting"):
        student_inputs = {k: v.to(device) for k, v in batch['student'].items()}
        teacher_inputs = {k: v.to(device) for k, v in batch['teacher'].items()}
        
        # Forward passes
        nomic_vecs = nomic_base(**student_inputs).last_hidden_state[:, 0, :].cpu().numpy()
        teacher_vecs = languagebind_base(**teacher_inputs).last_hidden_state[:, 0, :].cpu().numpy()
        
        nomic_list.append(nomic_vecs)
        teacher_list.append(teacher_vecs)

# Concatenate into massive matrices
all_nomic = np.vstack(nomic_list)
all_teacher = np.vstack(teacher_list)

# Save to Kaggle's local disk memory
np.save("/kaggle/working/nomic_features.npy", all_nomic)
np.save("/kaggle/working/teacher_features.npy", all_teacher)

print(f"Saved matrices! Shapes: Nomic {all_nomic.shape} | Teacher {all_teacher.shape}")

# CRITICAL: Clean up GPU memory entirely
del nomic_base
del languagebind_base
torch.cuda.empty_cache()

Extracting features on: cuda


config.json: 0.00B [00:00, ?B/s]

configuration_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Extracting frozen embeddings to disk (One-time pass)...


Extracting:   0%|          | 0/15625 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import TensorDataset

# Load vectors straight into CPU memory
nomic_features = torch.from_numpy(np.load("/kaggle/working/nomic_features.npy"))
teacher_features = torch.from_numpy(np.load("/kaggle/working/teacher_features.npy"))

# Bundle tensors together
fast_dataset = TensorDataset(nomic_features, teacher_features)

# Simple, ultra-fast DataLoader (No tokenizers needed anymore!)
fast_dataloader = DataLoader(
    fast_dataset, 
    batch_size=512,  # We can use a massive batch size now because VRAM is free!
    shuffle=True
)

In [ ]:
model = ProjectionModel(input_dim=768, hidden_dim=2048, output_dim=768).to(device)
criterion = CosineLoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

model.train()

print("Beginning ultra-fast alignment iterations...")
for epoch in range(EPOCHS):
    total_loss = 0.0
    progress_bar = tqdm(fast_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for nomic_vecs, teacher_vecs in progress_bar:
        nomic_vecs = nomic_vecs.to(device)
        teacher_vecs = teacher_vecs.to(device)
        
        # L2 Normalize target vectors
        teacher_vecs = F.normalize(teacher_vecs, p=2, dim=1)
        
        # Forward pass through MLP
        student_preds = model(nomic_vecs)
        student_preds = F.normalize(student_preds, p=2, dim=1)
        
        # Loss & Backward
        loss = criterion(student_preds, teacher_vecs)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    print(f"=== Epoch {epoch+1} Complete | Avg Loss: {total_loss / len(fast_dataloader):.4f} ===")

# Save the completed model weights
torch.save(model.state_dict(), "/kaggle/working/nomic_to_languagebind_mlp.pth")
print("[SAVE] Training complete and weights saved!")